In [0]:

# Verify uploaded dataset in Unity Catalog Volume

ROOT = "/Volumes/bbim707max/bbim707max/bbim707max"

print("Files available in the Databricks volume:")

display(dbutils.fs.ls(ROOT))


Files available in the Databricks volume:


path,name,size,modificationTime
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/taxi_zone_lookup.csv,taxi_zone_lookup.csv,12331,1793667386000
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/yellow_tripdata_2025-01.parquet,yellow_tripdata_2025-01.parquet,59158238,1793667399000


In [0]:

# BBIM707 – Part B Task 3
# Step 1: NYC Taxi Dataset Ingestion and Validation

from pyspark.sql import functions as F

# Actual Unity Catalog volume from your screenshot
ROOT = "/Volumes/bbim707max/bbim707max/bbim707max"

TAXI_PATH = f"{ROOT}/yellow_tripdata_2025-01.parquet"
ZONE_PATH = f"{ROOT}/taxi_zone_lookup.csv"

# Load the two datasets
taxi_df = spark.read.parquet(TAXI_PATH)

zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ZONE_PATH)
)

# Verify the Spark environment
print("===== SPARK ENVIRONMENT =====")
print("Spark Version:", spark.version)

# Verify dataset dimensions
print("\n===== DATASET VERIFICATION =====")
print("Taxi Records:", taxi_df.count())
print("Taxi Columns:", len(taxi_df.columns))

print("Zone Records:", zones_df.count())
print("Zone Columns:", len(zones_df.columns))

# Examine schemas
print("\n===== TAXI DATASET SCHEMA =====")
taxi_df.printSchema()

print("\n===== TAXI ZONE SCHEMA =====")
zones_df.printSchema()

# Preview datasets
print("\n===== NYC TAXI SAMPLE =====")
display(taxi_df.limit(5))

print("\n===== TAXI ZONE SAMPLE =====")
display(zones_df.limit(5))


===== SPARK ENVIRONMENT =====
Spark Version: 4.2.0

===== DATASET VERIFICATION =====
Taxi Records: 3475226
Taxi Columns: 20
Zone Records: 265
Zone Columns: 4

===== TAXI DATASET SCHEMA =====
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: dou

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0



===== TAXI ZONE SAMPLE =====


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone


In [0]:

# BBIM707 – Part B Task 3
# Step 2A: Data Cleaning and Feature Engineering

from pyspark.sql import functions as F

# Calculate journey duration from pickup and drop-off times
taxi_clean = (
    taxi_df
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime") -
            F.unix_timestamp("tpep_pickup_datetime")
        ) / 60.0
    )
    .select(
        F.col("trip_distance").cast("double"),
        F.col("fare_amount").cast("double"),
        F.col("trip_duration_minutes").cast("double")
    )
    .dropna()
    .filter(
        (F.col("trip_distance") > 0) &
        (F.col("trip_distance") <= 100) &
        (F.col("fare_amount") > 0) &
        (F.col("fare_amount") <= 500) &
        (F.col("trip_duration_minutes") > 0) &
        (F.col("trip_duration_minutes") <= 300)
    )
)

# Verify cleaned dataset
original_count = taxi_df.count()
clean_count = taxi_clean.count()
removed_count = original_count - clean_count

print("===== DATA CLEANING RESULTS =====")
print("Original Records:", original_count)
print("Cleaned Records:", clean_count)
print("Removed Records:", removed_count)
print("Retention Percentage:",
      round(clean_count / original_count * 100, 2), "%")

print("\n===== CLEANED DATASET SCHEMA =====")
taxi_clean.printSchema()

display(taxi_clean.limit(10))


===== DATA CLEANING RESULTS =====
Original Records: 3475226
Cleaned Records: 3251161
Removed Records: 224065
Retention Percentage: 93.55 %

===== CLEANED DATASET SCHEMA =====
root
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)



trip_distance,fare_amount,trip_duration_minutes
1.6,10.0,8.35
0.5,5.1,2.55
0.6,5.1,1.95
0.52,7.2,5.566666666666666
0.66,5.8,3.533333333333333
2.63,19.1,20.033333333333335
0.4,4.4,1.4666666666666666
1.6,12.1,12.4
2.8,19.1,19.666666666666668
1.71,11.4,9.566666666666666


In [0]:

# Step 2B: Validate the cleaned dataset

features = [
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes"
]

# Check null and NaN values
null_checks = taxi_clean.select([
    F.sum(
        F.when(
            F.col(c).isNull() | F.isnan(F.col(c)),
            1
        ).otherwise(0)
    ).alias(c)
    for c in features
])

print("===== MISSING VALUE VALIDATION =====")
display(null_checks)

print("===== DESCRIPTIVE STATISTICS =====")
display(taxi_clean.describe())

# Check processing partitions (DataFrame API - RDDs not supported on serverless)
partition_count = (
    taxi_clean
    .select(F.spark_partition_id().alias("partition_id"))
    .distinct()
    .count()
)
print("Spark DataFrame Partitions:", partition_count)


===== MISSING VALUE VALIDATION =====


trip_distance,fare_amount,trip_duration_minutes
0,0,0


===== DESCRIPTIVE STATISTICS =====


summary,trip_distance,fare_amount,trip_duration_minutes
count,3251161,3251161,3251161
mean,3.1676008785775998,17.945182696267917,14.682992824205282
stddev,4.143023488439146,16.205163839995947,11.420706095749251
min,0.01,0.01,0.016666666666666666
max,98.5,500.0,296.53333333333336


Spark DataFrame Partitions: 1


In [0]:

# Step 2C: Save cleaned dataset as Parquet

CLEAN_PATH = f"{ROOT}/cleaned_taxi_data"

(
    taxi_clean.write
    .mode("overwrite")
    .parquet(CLEAN_PATH)
)

# Reload to verify successful storage
cleaned_df = spark.read.parquet(CLEAN_PATH)

print("Saved cleaned records:", cleaned_df.count())
print("Cleaned data location:", CLEAN_PATH)

display(cleaned_df.limit(5))


Saved cleaned records: 3251161
Cleaned data location: /Volumes/bbim707max/bbim707max/bbim707max/cleaned_taxi_data


trip_distance,fare_amount,trip_duration_minutes
1.6,10.0,8.35
0.5,5.1,2.55
0.6,5.1,1.95
0.52,7.2,5.566666666666666
0.66,5.8,3.533333333333333


In [0]:

# BBIM707 – Part B Task 3
# Step 3A: Feature Engineering and Standardisation

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# Load cleaned dataset from Step 2
cleaned_df = spark.read.parquet(CLEAN_PATH)

feature_columns = [
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes"
]

# Split data before training to avoid evaluation leakage
training_df, validation_df = cleaned_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Sample training records to keep model training manageable
training_sample = training_df.sample(
    withReplacement=False,
    fraction=0.10,
    seed=42
).cache()

print("Training Sample Records:", training_sample.count())
print("Validation Records:", validation_df.count())

# Combine numerical columns into a feature vector
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="raw_features",
    handleInvalid="error"
)

# Standardise numerical features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Create preprocessing pipeline
feature_pipeline = Pipeline(
    stages=[assembler, scaler]
)

# Fit only on training data
feature_model = feature_pipeline.fit(training_sample)

# Apply same transformations to both datasets
train_features = feature_model.transform(training_sample)
validation_features = feature_model.transform(validation_df)

print("Feature Standardisation Completed")

display(
    train_features.select(
        *feature_columns,
        "features"
    ).limit(5)
)


Training Sample Records: 260111
Validation Records: 649347
Feature Standardisation Completed


trip_distance,fare_amount,trip_duration_minutes,features
0.01,0.05,0.4,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.7606049126819058"",""-1.097305912689741"",""-1.2515310317796955""]}"
0.01,0.09,5.883333333333334,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.7606049126819058"",""-1.0948598711881639"",""-0.7719228173099518""]}"
0.01,0.74,4.016666666666667,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.7606049126819058"",""-1.0551116967875325"",""-0.9351936988315667""]}"
0.01,0.85,10.166666666666666,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.7606049126819058"",""-1.0483850826581946"",""-0.39727445524696064""]}"
0.01,2.3,34.06666666666667,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.7606049126819058"",""-0.9597160782260167"",""1.693175938520859""]}"


In [0]:

# BBIM707 – Part B Task 3
# Step 3C: K-Means Cluster Profiling and Interpretation

from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Step 3B: Train K-Means models and select the best k (silhouette score)
print("===== K-MEANS MODEL SELECTION =====")
models = {}
silhouette_scores = {}
for k in (2, 3, 4, 5):
    model = KMeans(
        k=k,
        seed=42,
        featuresCol="features"
    ).fit(train_features)
    score = ClusteringEvaluator(
        featuresCol="features",
        predictionCol="prediction",
        metricName="silhouette"
    ).evaluate(model.transform(train_features))
    models[k] = model
    silhouette_scores[k] = score
    print(f"k = {k}, silhouette = {score:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
best_model = models[best_k]
print("Selected Number of Clusters:", best_k)

# Apply the selected model to validation data
cluster_predictions = best_model.transform(
    validation_features
)

# Calculate cluster characteristics
cluster_summary = (
    cluster_predictions
    .groupBy("prediction")
    .agg(
        F.count("*").alias("Total_Trips"),
        F.round(
            F.avg("trip_distance"), 2
        ).alias("Average_Distance_Miles"),
        F.round(
            F.avg("fare_amount"), 2
        ).alias("Average_Fare_USD"),
        F.round(
            F.avg("trip_duration_minutes"), 2
        ).alias("Average_Duration_Minutes")
    )
    .orderBy("prediction")
)

print("===== SELECTED K-MEANS MODEL =====")
print("Selected Number of Clusters:", best_k)

print("\n===== CLUSTER CHARACTERISTICS =====")
display(cluster_summary)

# Cluster distribution
cluster_distribution = (
    cluster_predictions
    .groupBy("prediction")
    .count()
    .withColumnRenamed("count", "Trip_Count")
    .orderBy("prediction")
)

print("\n===== CLUSTER DISTRIBUTION =====")
display(cluster_distribution)

# Verify predictions
print("\n===== PREDICTION VALIDATION =====")
print("Total Validation Records:",
      validation_features.count())

print("Total Predicted Records:",
      cluster_predictions.count())

print("Distinct Clusters:",
      cluster_predictions.select("prediction")
      .distinct().count())


===== K-MEANS MODEL SELECTION =====
k = 2, silhouette = 0.8819
k = 3, silhouette = 0.7555
k = 4, silhouette = 0.7450
k = 5, silhouette = 0.6728
Selected Number of Clusters: 2
===== SELECTED K-MEANS MODEL =====
Selected Number of Clusters: 2

===== CLUSTER CHARACTERISTICS =====


prediction,Total_Trips,Average_Distance_Miles,Average_Fare_USD,Average_Duration_Minutes
0,581340,1.99,13.51,11.91
1,68007,13.3,55.92,38.52



===== CLUSTER DISTRIBUTION =====


prediction,Trip_Count
0,581340
1,68007



===== PREDICTION VALIDATION =====
Total Validation Records: 649347
Total Predicted Records: 649347
Distinct Clusters: 2


In [0]:

# Step 3C – Independent Validation of Selected K-Means Model

from pyspark.ml.evaluation import ClusteringEvaluator

# Use a reproducible subset of held-out validation records
validation_evaluation = validation_features.sample(
    withReplacement=False,
    fraction=0.10,
    seed=99
)

validation_predictions = best_model.transform(
    validation_evaluation
)

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

validation_score = evaluator.evaluate(
    validation_predictions
)

print("===== INDEPENDENT MODEL EVALUATION =====")
print("Selected K:", best_k)
print("Training Silhouette:", round(silhouette_scores[best_k], 4))
print("Validation Silhouette:", round(validation_score, 4))
print("Validation Sample Records:", validation_evaluation.count())


===== INDEPENDENT MODEL EVALUATION =====
Selected K: 2
Training Silhouette: 0.8819
Validation Silhouette: 0.8813
Validation Sample Records: 64943


In [0]:

# Step 3D – Save K-Means Model for Streaming Predictions

FEATURE_MODEL_PATH = f"{ROOT}/models/feature_pipeline"
KMEANS_MODEL_PATH = f"{ROOT}/models/kmeans_best"

feature_model.write().overwrite().save(FEATURE_MODEL_PATH)
best_model.write().overwrite().save(KMEANS_MODEL_PATH)

print("===== MODEL SAVED SUCCESSFULLY =====")
print("Selected K:", best_k)
print("Feature Pipeline:", FEATURE_MODEL_PATH)
print("K-Means Model:", KMEANS_MODEL_PATH)

display(dbutils.fs.ls(f"{ROOT}/models"))


===== MODEL SAVED SUCCESSFULLY =====
Selected K: 2
Feature Pipeline: /Volumes/bbim707max/bbim707max/bbim707max/models/feature_pipeline
K-Means Model: /Volumes/bbim707max/bbim707max/bbim707max/models/kmeans_best


path,name,size,modificationTime
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/models/feature_pipeline/,feature_pipeline/,0,1793669081712
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/models/kmeans_best/,kmeans_best/,0,1793669081712


In [0]:

# BBIM707 – Part B Task 3
# Step 4A: Configure Streaming Directories

ROOT = "/Volumes/bbim707max/bbim707max/bbim707max"

STAGING_PATH = f"{ROOT}/stream_staging"
STREAM_INPUT_PATH = f"{ROOT}/stream_input"
CHECKPOINT_PATH = f"{ROOT}/checkpoints/kmeans_stream_v1"

# Create required directories
dbutils.fs.mkdirs(STAGING_PATH)
dbutils.fs.mkdirs(STREAM_INPUT_PATH)

print("===== STREAMING CONFIGURATION =====")
print("Staging Directory:", STAGING_PATH)
print("Streaming Input:", STREAM_INPUT_PATH)
print("Checkpoint Directory:", CHECKPOINT_PATH)

print("\nStreaming directories configured successfully.")


===== STREAMING CONFIGURATION =====
Staging Directory: /Volumes/bbim707max/bbim707max/bbim707max/stream_staging
Streaming Input: /Volumes/bbim707max/bbim707max/bbim707max/stream_input
Checkpoint Directory: /Volumes/bbim707max/bbim707max/bbim707max/checkpoints/kmeans_stream_v1

Streaming directories configured successfully.


In [0]:

# Step 4B: Prepare Incremental Taxi Data Batches

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Select 10,000 cleaned records
stream_source = cleaned_df.limit(10000)

# Assign deterministic row numbers for this demonstration
window_spec = Window.orderBy(
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes"
)

numbered_df = (
    stream_source
    .withColumn(
        "event_id",
        F.row_number().over(window_spec)
    )
    .withColumn(
        "batch_id",
        F.floor(
            (F.col("event_id") - 1) / 1000
        ).cast("integer")
    )
)

# Save prepared data to a separate staging location
(
    numbered_df.write
    .mode("overwrite")
    .parquet(STAGING_PATH)
)

staged_df = spark.read.parquet(STAGING_PATH)

print("===== STREAMING DATA PREPARATION =====")
print("Total Prepared Records:", staged_df.count())

display(
    staged_df.groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


===== STREAMING DATA PREPARATION =====
Total Prepared Records: 10000


batch_id,count
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,1000
9,1000


In [0]:

# Step 4C: Validate Streaming Input Data

print("===== STREAMING DATA SCHEMA =====")
staged_df.printSchema()

print("\n===== STREAMING FEATURE VALIDATION =====")

required_columns = [
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes",
    "event_id",
    "batch_id"
]

missing_columns = [
    c for c in required_columns
    if c not in staged_df.columns
]

print("Missing Required Columns:", missing_columns)

assert not missing_columns, "Required columns missing"

assert staged_df.count() == 10000, (
    "Expected 10,000 prepared records"
)

print("Streaming data validation successful.")

display(staged_df.limit(10))


===== STREAMING DATA SCHEMA =====
root
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- event_id: integer (nullable = true)
 |-- batch_id: integer (nullable = true)


===== STREAMING FEATURE VALIDATION =====
Missing Required Columns: []
Streaming data validation successful.


trip_distance,fare_amount,trip_duration_minutes,event_id,batch_id
0.01,3.0,0.16666666666666666,1,0
0.01,3.0,0.16666666666666666,2,0
0.01,3.0,0.26666666666666666,3,0
0.01,3.0,0.55,4,0
0.01,3.0,0.9166666666666666,5,0
0.01,3.0,1.2166666666666666,6,0
0.01,3.7,1.3333333333333333,7,0
0.01,23.0,0.11666666666666667,8,0
0.01,70.0,0.11666666666666667,9,0
0.02,3.0,0.1,10,0


In [0]:

# BBIM707 – Part B Task 3
# Step 5A: Load Trained Models and Prepare Streaming Output

from pyspark.ml import PipelineModel
from pyspark.ml.clustering import KMeansModel
from pyspark.sql import functions as F
from delta.tables import DeltaTable

ROOT = "/Volumes/bbim707max/bbim707max/bbim707max"

STAGING_PATH = f"{ROOT}/stream_staging"
STREAM_INPUT_PATH = f"{ROOT}/stream_input"
CHECKPOINT_PATH = f"{ROOT}/checkpoints/kmeans_stream_v1"

OUTPUT_TABLE = (
    "bbim707max.bbim707max.taxi_stream_predictions"
)

# Load models saved in Step 3D
feature_model = PipelineModel.load(
    f"{ROOT}/models/feature_pipeline"
)

best_model = KMeansModel.load(
    f"{ROOT}/models/kmeans_best"
)

# Load prepared batches
staged_df = spark.read.parquet(STAGING_PATH)

# Prepare output schema using the trained model
empty_input = staged_df.limit(0)

empty_predictions = best_model.transform(
    feature_model.transform(empty_input)
).select(
    "event_id",
    "batch_id",
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes",
    F.col("prediction").cast("integer").alias("cluster")
)

# Create the output table if needed
if not spark.catalog.tableExists(OUTPUT_TABLE):
    (
        empty_predictions.write
        .format("delta")
        .mode("error")
        .saveAsTable(OUTPUT_TABLE)
    )

print("===== STREAMING MODEL CONFIGURATION =====")
print("K-Means clusters:", best_model.getK())
print("Prepared records:", staged_df.count())
print("Prediction table:", OUTPUT_TABLE)

print("\nSaved models loaded successfully.")


===== STREAMING MODEL CONFIGURATION =====
K-Means clusters: 2
Prepared records: 10000
Prediction table: bbim707max.bbim707max.taxi_stream_predictions

Saved models loaded successfully.


In [0]:

# BBIM707 – Part B Task 3
# Step 6A: Simulate Arrival of First Five Taxi Batches

from pyspark.sql import functions as F

# Confirm that the experiment has not started
existing_files = dbutils.fs.ls(STREAM_INPUT_PATH)
existing_predictions = spark.table(OUTPUT_TABLE).count()

assert len(existing_files) == 0, (
    "Streaming input already contains files. "
    "Check existing data before proceeding."
)

assert existing_predictions == 0, (
    "Prediction table already contains records."
)

# Load the prepared streaming dataset
staged_df = spark.read.parquet(STAGING_PATH)

print("===== SIMULATED REAL-TIME DATA ARRIVAL =====")

# Add batches 0–4
for i in range(5):

    batch_df = staged_df.filter(
        F.col("batch_id") == i
    )

    batch_count = batch_df.count()

    # Each batch is written to a separate directory
    (
        batch_df.write
        .mode("append")
        .parquet(STREAM_INPUT_PATH)
    )

    print(
        f"Batch {i} added: {batch_count} taxi records"
    )

print("\nFirst five batches added successfully.")


===== SIMULATED REAL-TIME DATA ARRIVAL =====
Batch 0 added: 1000 taxi records
Batch 1 added: 1000 taxi records
Batch 2 added: 1000 taxi records
Batch 3 added: 1000 taxi records
Batch 4 added: 1000 taxi records

First five batches added successfully.


In [0]:

# BBIM707 – Part B Task 3
# Step 6B: Execute Spark Structured Streaming

from pyspark.sql import functions as F

# Step 5C: Streaming query configuration (defines run_taxi_stream)
def run_taxi_stream():
    # Streaming Parquet reads require an explicit schema
    stream_schema = staged_df.schema

    streaming_input = (
        spark.readStream
        .schema(stream_schema)
        .parquet(STREAM_INPUT_PATH)
    )

    predictions = (
        best_model.transform(
            feature_model.transform(streaming_input)
        )
        .select(
            "event_id",
            "batch_id",
            "trip_distance",
            "fare_amount",
            "trip_duration_minutes",
            F.col("prediction").cast("integer").alias("cluster")
        )
    )

    query = (
        predictions.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(OUTPUT_TABLE)
    )

    query.awaitTermination()
    return query

print("===== STARTING SPARK STRUCTURED STREAMING =====")

# Run the streaming query configured in Step 5C
query1 = run_taxi_stream()

print("\n===== STREAMING EXECUTION RESULTS =====")

# Verify predicted records
predicted_count = spark.table(OUTPUT_TABLE).count()

print("Total Processed Records:", predicted_count)

# Display cluster distribution
display(
    spark.table(OUTPUT_TABLE)
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)

# Display streaming progress
print("\n===== STREAMING PROGRESS =====")
print(query1.lastProgress)


===== STARTING SPARK STRUCTURED STREAMING =====

===== STREAMING EXECUTION RESULTS =====
Total Processed Records: 5000


cluster,count
0,4999
1,1



===== STREAMING PROGRESS =====
{
    "id": "ee8cb8bd-75ee-4ec5-b5d0-bc2f738c586c",
    "runId": "66d60627-6014-4413-a81b-d9a1d84b4dad",
    "name": null,
    "timestamp": "2026-11-03T01:42:08.227Z",
    "batchId": 0,
    "batchDuration": 3326,
    "durationMs": {
        "triggerExecution": 3318,
        "queryPlanning": 394,
        "collectSourceMetrics": 2,
        "getBatch": 435,
        "commitOffsets": 116,
        "addBatch": 1478,
        "latestOffset": 128,
        "commitBatch": 489,
        "walCommit": 145
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[dbfs:/Volumes/bbim707max/bbim707max/bbim707max/stream_input]",
            "startOffset": null,
            "endOffset": "{\"logOffset\":0}",
            "latestOffset": null,
            "numInputRows": 5000,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 1503.307276007216,
            "metrics": {}
        }
    ]

In [0]:

# BBIM707 – Part B Task 3
# Step 6C: Streaming Prediction Validation

from pyspark.sql import functions as F

predictions_df = spark.table(OUTPUT_TABLE)

total_predictions = predictions_df.count()

unique_events = (
    predictions_df
    .select("event_id")
    .distinct()
    .count()
)

print("===== STREAMING PREDICTION VALIDATION =====")
print("Expected Records:", 5000)
print("Actual Predictions:", total_predictions)
print("Unique Event IDs:", unique_events)

print(
    "Record Completeness:",
    total_predictions == 5000
)

print(
    "No Duplicate Predictions:",
    total_predictions == unique_events
)

print("\n===== CLUSTER CHARACTERISTICS =====")

display(
    predictions_df
    .groupBy("cluster")
    .agg(
        F.count("*").alias("Trip_Count"),
        F.round(
            F.avg("trip_distance"), 2
        ).alias("Average_Distance_Miles"),
        F.round(
            F.avg("fare_amount"), 2
        ).alias("Average_Fare_USD"),
        F.round(
            F.avg("trip_duration_minutes"), 2
        ).alias("Average_Duration_Minutes")
    )
    .orderBy("cluster")
)


===== STREAMING PREDICTION VALIDATION =====
Expected Records: 5000
Actual Predictions: 5000
Unique Event IDs: 5000
Record Completeness: True
No Duplicate Predictions: True

===== CLUSTER CHARACTERISTICS =====


cluster,Trip_Count,Average_Distance_Miles,Average_Fare_USD,Average_Duration_Minutes
0,4999,1.13,9.77,8.53
1,1,1.42,40.1,52.43


In [0]:

# BBIM707 – Part B Task 3
# Step 7A: Add Remaining Streaming Batches

from pyspark.sql import functions as F

staged_df = spark.read.parquet(STAGING_PATH)

print("===== SECOND STREAMING DATA ARRIVAL =====")

# Verify first run completed
current_count = spark.table(OUTPUT_TABLE).count()

assert current_count == 5000, (
    f"Expected 5000 existing predictions, found {current_count}"
)

# Check whether remaining batches have already been added
existing_input = dbutils.fs.ls(STREAM_INPUT_PATH)

print("Existing input files:", len(existing_input))
print("Existing predictions:", current_count)

# Introduce batches 5–9
for i in range(5, 10):

    batch_df = staged_df.filter(
        F.col("batch_id") == i
    )

    batch_count = batch_df.count()

    (
        batch_df.write
        .mode("append")
        .parquet(STREAM_INPUT_PATH)
    )

    print(f"Batch {i} added: {batch_count} records")

print("\nRemaining five batches added successfully.")


===== SECOND STREAMING DATA ARRIVAL =====
Existing input files: 16
Existing predictions: 5000
Batch 5 added: 1000 records
Batch 6 added: 1000 records
Batch 7 added: 1000 records
Batch 8 added: 1000 records
Batch 9 added: 1000 records

Remaining five batches added successfully.


In [0]:

# BBIM707 – Part B Task 3
# Step 7B: Checkpoint-Based Streaming Continuation

print("===== RESTARTING STRUCTURED STREAMING =====")

# Resume using the SAME checkpoint
query2 = run_taxi_stream()

print("\n===== SECOND STREAMING RESULTS =====")

final_count = spark.table(OUTPUT_TABLE).count()

print("Expected Total Records:", 10000)
print("Actual Total Predictions:", final_count)

print("\n===== FINAL CLUSTER DISTRIBUTION =====")

display(
    spark.table(OUTPUT_TABLE)
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)

print("\n===== STREAMING PROGRESS =====")
print(query2.lastProgress)


===== RESTARTING STRUCTURED STREAMING =====

===== SECOND STREAMING RESULTS =====
Expected Total Records: 10000
Actual Total Predictions: 10000

===== FINAL CLUSTER DISTRIBUTION =====


cluster,count
0,9078
1,922



===== STREAMING PROGRESS =====
{
    "id": "ee8cb8bd-75ee-4ec5-b5d0-bc2f738c586c",
    "runId": "c88f52d6-99e5-4a34-b4db-529435290650",
    "name": null,
    "timestamp": "2026-11-03T02:00:09.008Z",
    "batchId": 1,
    "batchDuration": 2823,
    "durationMs": {
        "triggerExecution": 2822,
        "queryPlanning": 245,
        "collectSourceMetrics": 0,
        "getBatch": 391,
        "commitOffsets": 139,
        "addBatch": 1283,
        "latestOffset": 152,
        "commitBatch": 447,
        "walCommit": 69
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[dbfs:/Volumes/bbim707max/bbim707max/bbim707max/stream_input]",
            "startOffset": "{\"logOffset\":0}",
            "endOffset": "{\"logOffset\":1}",
            "latestOffset": null,
            "numInputRows": 5000,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 1771.1654268508678,
            "metrics": {}


In [0]:

# BBIM707 – Part B Task 3
# Step 7C: Final Data Consistency and Checkpoint Verification

from pyspark.sql import functions as F

final_df = spark.table(OUTPUT_TABLE)

# Verify complete processing
total_records = final_df.count()
unique_events = final_df.select("event_id").distinct().count()

# Check null cluster predictions
null_predictions = final_df.filter(
    F.col("cluster").isNull()
).count()

# Verify batch distribution
batch_counts = (
    final_df.groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

print("===== FINAL STREAMING VALIDATION =====")
print("Expected Records:", 10000)
print("Actual Predictions:", total_records)
print("Unique Event IDs:", unique_events)
print("Missing Cluster Predictions:", null_predictions)

print("Complete Processing:", total_records == 10000)
print("No Duplicate Events:", total_records == unique_events)
print("No Missing Predictions:", null_predictions == 0)

print("\n===== BATCH COMPLETENESS =====")
display(batch_counts)

print("\n===== CHECKPOINT DIRECTORY =====")
display(dbutils.fs.ls(CHECKPOINT_PATH))

print("\n===== FINAL CLUSTER DISTRIBUTION =====")
display(
    final_df.groupBy("cluster")
    .count()
    .orderBy("cluster")
)


===== FINAL STREAMING VALIDATION =====
Expected Records: 10000
Actual Predictions: 10000
Unique Event IDs: 10000
Missing Cluster Predictions: 0
Complete Processing: True
No Duplicate Events: True
No Missing Predictions: True

===== BATCH COMPLETENESS =====


batch_id,count
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,1000
9,1000



===== CHECKPOINT DIRECTORY =====


path,name,size,modificationTime
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/checkpoints/kmeans_stream_v1/commits/,commits/,0,1793671324970
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/checkpoints/kmeans_stream_v1/metadata,metadata,45,1793670127000
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/checkpoints/kmeans_stream_v1/offsets/,offsets/,0,1793671324970
dbfs:/Volumes/bbim707max/bbim707max/bbim707max/checkpoints/kmeans_stream_v1/sources/,sources/,0,1793671324970



===== FINAL CLUSTER DISTRIBUTION =====


cluster,count
0,9078
1,922


In [0]:

# BBIM707 Task 3 - Combined Streaming and K-Means Test

from pyspark.ml import PipelineModel
from pyspark.ml.clustering import KMeansModel
from pyspark.sql.functions import (
    col, lit, current_timestamp, rand
)

ROOT = "/Volumes/bbim707max/bbim707max/bbim707max"

# 1. Load existing models
feature_model = PipelineModel.load(
    f"{ROOT}/models/feature_pipeline"
)

best_model = KMeansModel.load(
    f"{ROOT}/models/kmeans_best"
)

print("Models loaded successfully.")
print("Selected K:", best_model.getK())

# 2. Generate streaming taxi-like events
# Synthetic values for testing, not actual live taxi trips
live_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

taxi_stream = live_stream.select(
    col("value").alias("event_id"),
    col("timestamp").alias("event_time"),
    (rand(seed=42) * 15 + 1)
        .alias("trip_distance"),
    (rand(seed=43) * 60 + 5)
        .alias("fare_amount"),
    (rand(seed=44) * 45 + 2)
        .alias("trip_duration_minutes")
)

# 3. Apply existing feature preprocessing
processed_stream = feature_model.transform(
    taxi_stream
)

# 4. Apply trained K-Means clustering
predictions = best_model.transform(
    processed_stream
)

# 5. Start streaming query
# An explicit checkpoint location is required in this workspace;
# serverless compute can only write checkpoints to a UC volume.
spark.sql(
    "CREATE VOLUME IF NOT EXISTS workspace.default.streaming_checkpoints"
)

# A one-shot rate-source query cannot resume from a previous
# checkpoint; clear previous test state to start over.
CHECKPOINT_DIR = (
    "/Volumes/workspace/default/streaming_checkpoints/"
    "bbim707_kmeans_live_demo"
)
dbutils.fs.rm(CHECKPOINT_DIR, recurse=True)

query = (
    predictions.select(
        "event_id",
        "event_time",
        "trip_distance",
        "fare_amount",
        "trip_duration_minutes",
        "prediction"
    )
    .writeStream
    .format("memory")
    .queryName("bbim707_kmeans_live_demo")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

# 6. Display results
print("===== STREAMING CLUSTER PREDICTIONS =====")

display(spark.sql("""
    SELECT *
    FROM bbim707_kmeans_live_demo
    ORDER BY event_id
    LIMIT 20
"""))

print("===== CLUSTER DISTRIBUTION =====")

display(spark.sql("""
    SELECT
        prediction,
        COUNT(*) AS total_records
    FROM bbim707_kmeans_live_demo
    GROUP BY prediction
    ORDER BY prediction
"""))

print("Streaming query completed successfully.")


Models loaded successfully.
Selected K: 2
===== STREAMING CLUSTER PREDICTIONS =====


event_id,event_time,trip_distance,fare_amount,trip_duration_minutes,prediction
0,2026-11-04T20:56:42.873Z,8.806336270310048,21.01271208212377,32.8703566606079,1
1,2026-11-04T20:56:42.973Z,15.438732758554732,7.261645447788741,12.744425662648766,0
2,2026-11-04T20:56:43.073Z,3.605625046945859,23.236470673582367,21.32027103473137,0
3,2026-11-04T20:56:43.173Z,9.44116685915383,12.26070003056389,15.615874249688394,0


===== CLUSTER DISTRIBUTION =====


prediction,total_records
0,3
1,1


Streaming query completed successfully.
